# QOLDA Vision: reproducible public-data benchmark
Run all cells in Colab (CPU). Downloads UCI405; trains MLP and baselines; writes per-fold reports, checkpoints, ablations and figures. No personal data upload required. This is public Vicon data, not webcam or QOLDA data. Dataset description says 12 users but downloaded file has 14 user IDs; see report. Typical execution may take several minutes.

In [ ]:
%pip -q install numpy scipy scikit-learn matplotlib torch
from pathlib import Path
Path("research/results").mkdir(parents=True,exist_ok=True)
Path("docs/assets").mkdir(parents=True,exist_ok=True)

In [ ]:
from pathlib import Path
Path("research").mkdir(exist_ok=True)
Path("docs/assets").mkdir(parents=True,exist_ok=True)
Path("research/benchmark.py").write_text('"""Real UCI405 subject-disjoint experiment. Run from repository root."""\nfrom pathlib import Path\nimport argparse, csv, hashlib, json, sys, platform, time, urllib.request, zipfile\nimport numpy as np\nfrom scipy.spatial.distance import pdist\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.neighbors import KNeighborsClassifier, NearestCentroid\nfrom sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score\nimport torch\nfrom torch import nn\n\nURL=\'https://archive.ics.uci.edu/static/public/405/motion+capture+hand+postures.zip\'\nFEATURES=[\'count\',\'mean_x\',\'mean_y\',\'mean_z\',\'std_x\',\'std_y\',\'std_z\',\'radius_q25\',\'radius_q50\',\'radius_q75\',\'pair_q10\',\'pair_q25\',\'pair_q50\',\'pair_q75\',\'pair_q90\',\'eig_1\',\'eig_2\',\'eig_3\']\n\ndef cloud_features(points):\n    """Permutation-invariant features: marker indices do not identify fingers."""\n    p=np.asarray(points,float); p=p[np.isfinite(p).all(axis=1)]\n    if len(p)<3: raise ValueError(\'At least three visible markers required\')\n    radii=np.linalg.norm(p,axis=1); ev=np.linalg.eigvalsh(np.cov(p.T))\n    return np.r_[len(p),p.mean(0),p.std(0),np.quantile(radii,[.25,.5,.75]),np.quantile(pdist(p),[.1,.25,.5,.75,.9]),ev]\n\ndef fetch(path):\n    path.parent.mkdir(parents=True,exist_ok=True)\n    if not path.exists():\n        archive=path.parent/\'uci405.zip\';urllib.request.urlretrieve(URL,archive)\n        with zipfile.ZipFile(archive) as z: path.write_bytes(z.read(\'Postures.csv\'))\n    return path\n\ndef load_data(path,cap,seed):\n    a=np.genfromtxt(path,delimiter=\',\',skip_header=1,missing_values=\'?\',filling_values=np.nan)\n    original=a[a[:,0]>0]; rng=np.random.default_rng(seed); indices=[]\n    for user in np.unique(original[:,1]):\n        for cls in range(1,6):\n            idx=np.flatnonzero((original[:,1]==user)&(original[:,0]==cls))\n            indices.extend(rng.choice(idx,min(cap,len(idx)),replace=False).tolist())\n    indices=np.array(sorted(indices));a=original[indices]\n    clouds=a[:,2:].reshape(-1,12,3)\n    return np.stack([cloud_features(p) for p in clouds]),a[:,0].astype(int),a[:,1].astype(int),clouds,indices,len(original)\n\ndef mlp_fit(x,y,seed,epochs):\n    torch.manual_seed(seed);torch.set_num_threads(1)\n    model=nn.Sequential(nn.Linear(x.shape[1],64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,5))\n    opt=torch.optim.Adam(model.parameters(),lr=.001,weight_decay=.0001)\n    tx=torch.tensor(x,dtype=torch.float32);ty=torch.tensor(y-1,dtype=torch.long)\n    for _ in range(epochs):\n        for ids in torch.randperm(len(tx)).split(128):\n            opt.zero_grad();loss=nn.functional.cross_entropy(model(tx[ids]),ty[ids]);loss.backward();opt.step()\n    model.eval();return model\n\ndef infer(model,x):\n    if isinstance(model,nn.Module):\n        with torch.no_grad():return model(torch.tensor(x,dtype=torch.float32)).argmax(1).numpy()+1\n    return model.predict(x)\n\ndef metrics(y,p):return {\'accuracy\':float(accuracy_score(y,p)),\'macro_f1\':float(f1_score(y,p,labels=[1,2,3,4,5],average=\'macro\',zero_division=0))}\n\ndef run(args):\n    start=time.time();out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    path=fetch(Path(args.data));x,y,u,clouds,indices,total=load_data(path,args.cap,args.seed)\n    users=np.unique(u);variants={\'full\':np.arange(18),\'no_count\':np.arange(1,18),\'no_pair_distances\':np.r_[0:10,15:18]}\n    folds=[];opens=[];robust=[];predictions=[];splits=[]\n    for j,test_user in enumerate(users):\n        val_user=users[(j+1)%len(users)]\n        tr=(u!=test_user)&(u!=val_user);va=u==val_user;te=u==test_user\n        splits.append({\'test_user\':int(test_user),\'validation_user\':int(val_user),\'train_users\':users[~np.isin(users,[test_user,val_user])].tolist()})\n        for variant,cols in variants.items():\n            scaler=StandardScaler().fit(x[tr][:,cols]);a=scaler.transform(x[tr][:,cols]);b=scaler.transform(x[te][:,cols])\n            models={\'centroid\':NearestCentroid().fit(a,y[tr]),\'knn\':KNeighborsClassifier(5).fit(a,y[tr]),\'mlp\':mlp_fit(a,y[tr],args.seed+j,args.epochs)}\n            for name,model in models.items():\n                p=infer(model,b);folds.append({\'test_user\':int(test_user),\'variant\':variant,\'model\':name,\'n_test\':int(te.sum()),**metrics(y[te],p)})\n                if variant==\'full\':\n                    predictions.extend({\'test_user\':int(test_user),\'source_index\':int(i),\'truth\':int(t),\'prediction\':int(g),\'model\':name} for i,t,g in zip(indices[te],y[te],p))\n                if name==\'mlp\' and variant==\'full\':\n                    torch.save({\'state_dict\':model.state_dict(),\'input_dim\':18,\'classes\':[1,2,3,4,5],\'scaler_mean\':scaler.mean_.tolist(),\'scaler_scale\':scaler.scale_.tolist(),\'test_user\':int(test_user),\'train_users\':splits[-1][\'train_users\']},out/f\'mlp_holdout_{test_user}.pt\')\n            if variant==\'full\':\n                rng=np.random.default_rng(args.seed+j)\n                for drop in [1,2]:\n                    degraded=[]\n                    for cloud in clouds[te]:\n                        visible=cloud[np.isfinite(cloud).all(1)];n=max(3,len(visible)-drop)\n                        degraded.append(cloud_features(visible[rng.choice(len(visible),n,replace=False)]))\n                    degraded=scaler.transform(np.asarray(degraded))\n                    for name,model in models.items():robust.append({\'test_user\':int(test_user),\'drop_markers\':drop,\'model\':name,**metrics(y[te],infer(model,degraded))})\n        # Unknown class 5 never enters training or threshold calibration.\n        known_tr=tr&(y!=5);known_va=va&(y!=5)\n        sc=StandardScaler().fit(x[known_tr]);kn=KNeighborsClassifier(5).fit(sc.transform(x[known_tr]),y[known_tr])\n        val_dist=kn.kneighbors(sc.transform(x[known_va]))[0].mean(1);threshold=float(np.quantile(val_dist,.95))\n        test_dist=kn.kneighbors(sc.transform(x[te]))[0].mean(1);pred=kn.predict(sc.transform(x[te]));unknown=y[te]==5;reject=test_dist>threshold;known=~unknown;accepted=known&~reject\n        opens.append({\'test_user\':int(test_user),\'threshold\':threshold,\'unknown_class\':5,\'unknown_recall\':float(reject[unknown].mean()) if unknown.any() else None,\'known_false_reject\':float(reject[known].mean()),\'unknown_auroc\':float(roc_auc_score(unknown,test_dist)) if unknown.any() and known.any() else None,\'known_accuracy_no_rejection\':float((pred[known]==y[te][known]).mean()),\'accepted_known_accuracy\':float((pred[accepted]==y[te][accepted]).mean()) if accepted.any() else None,\'accepted_known_n\':int(accepted.sum())})\n        print(f\'Finished held-out user ID {test_user}\',flush=True)\n    summary=[]\n    for variant in variants:\n        for model in [\'centroid\',\'knn\',\'mlp\']:\n            subset=[r for r in folds if r[\'variant\']==variant and r[\'model\']==model]\n            summary.append({\'variant\':variant,\'model\':model,\'accuracy_mean\':float(np.mean([r[\'accuracy\'] for r in subset])),\'accuracy_sd\':float(np.std([r[\'accuracy\'] for r in subset],ddof=1)),\'macro_f1_mean\':float(np.mean([r[\'macro_f1\'] for r in subset]))})\n    report={\'dataset\':\'UCI Motion Capture Hand Postures (405)\',\'doi\':\'10.24432/C5TG86\',\'license\':\'CC BY 4.0\',\'source_url\':URL,\'sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\'original_non_dummy_rows\':total,\'sampled_rows\':len(y),\'user_ids\':users.tolist(),\'source_description_participants\':12,\'metadata_discrepancy\':\'UCI description states 12 users; downloaded CSV contains 14 unique user IDs. Identity mapping unresolved.\',\'cap_per_user_class\':args.cap,\'seed\':args.seed,\'epochs\':args.epochs,\'features\':FEATURES,\'split\':\'one test user ID, next user ID validation, all others training; identical split for all models\',\'summary\':summary,\'folds\':folds,\'open_set\':opens,\'synthetic_marker_dropout\':robust,\'splits\':splits,\'runtime_seconds\':time.time()-start,\'versions\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'torch\':torch.__version__},\'limitations\':[\'Public Vicon marker data, not MediaPipe or QOLDA recordings.\',\'No original participant recruitment.\',\'Marker dropout is synthetic, not measured lighting or camera failure.\',\'No chronological timestamps; no temporal claims from this dataset.\',\'Fixed hyperparameters and one seed; no confidence claims from repeated tuning.\']}\n    (out/\'benchmark.json\').write_text(json.dumps(report,indent=2,allow_nan=False));(out/\'predictions.json\').write_text(json.dumps(predictions));(out/\'sample_indices.json\').write_text(json.dumps(indices.tolist()))\n    print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'research/data/Postures.csv\');p.add_argument(\'--output\',default=\'research/results\');p.add_argument(\'--cap\',type=int,default=75);p.add_argument(\'--seed\',type=int,default=42);p.add_argument(\'--epochs\',type=int,default=50);run(p.parse_args())\n')
Path("research/plot_results.py").write_text('from pathlib import Path\nimport json\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nr=json.loads(Path(\'research/results/benchmark.json\').read_text());out=Path(\'docs/assets\');out.mkdir(exist_ok=True)\nplt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'axes.spines.top\':False,\'axes.spines.right\':False,\'figure.facecolor\':\'#f6f7f3\',\'axes.facecolor\':\'#f6f7f3\'})\nmodels=[\'centroid\',\'knn\',\'mlp\'];colors=[\'#a6b5a1\',\'#69a88a\',\'#096b59\']\nfig,ax=plt.subplots(figsize=(9,4.5));base=[x for x in r[\'summary\'] if x[\'variant\']==\'full\'];ax.bar(models,[x[\'accuracy_mean\']*100 for x in base],yerr=[x[\'accuracy_sd\']*100 for x in base],color=colors,capsize=6)\nfor i,x in enumerate(base):ax.text(i,x[\'accuracy_mean\']*100+2,f"{x[\'accuracy_mean\']*100:.1f}%",ha=\'center\',fontweight=\'bold\')\nax.set(ylim=(0,105),ylabel=\'Accuracy (%)\',title=\'UCI hand postures · held-out user-ID evaluation\');fig.text(.1,.01,\'Mean ± SD across 14 IDs. Public Vicon data; not webcam or QOLDA measurements.\',fontsize=9);fig.tight_layout(rect=[0,.06,1,1]);fig.savefig(out/\'benchmark.png\',dpi=170);plt.close(fig)\nfig,ax=plt.subplots(figsize=(9,4.5));variants=[\'full\',\'no_count\',\'no_pair_distances\'];xx=np.arange(3)\nfor i,m in enumerate(models):ax.bar(xx+(i-1)*.23,[next(s for s in r[\'summary\'] if s[\'model\']==m and s[\'variant\']==v)[\'macro_f1_mean\'] for v in variants],width=.23,label=m,color=colors[i])\nax.set_xticks(xx,[\'All features\',\'Without marker count\',\'Without pair distances\']);ax.set(ylabel=\'Macro F1\',ylim=(0,1),title=\'Feature ablation · identical user-ID splits\');ax.legend();fig.tight_layout();fig.savefig(out/\'ablation.png\',dpi=170);plt.close(fig)\nfig,ax=plt.subplots(figsize=(9,4.5))\nfor m,c in zip(models,colors):\n values=[next(s for s in base if s[\'model\']==m)[\'accuracy_mean\']]+[np.mean([s[\'accuracy\'] for s in r[\'synthetic_marker_dropout\'] if s[\'model\']==m and s[\'drop_markers\']==d]) for d in [1,2]]\n ax.plot([0,1,2],np.array(values)*100,\'o-\',label=m,color=c)\nax.set(xticks=[0,1,2],xlabel=\'Artificially removed markers (minimum 3 retained)\',ylabel=\'Accuracy (%)\',title=\'Synthetic occlusion stress test · not measured camera failure\');ax.legend();fig.tight_layout();fig.savefig(out/\'robustness.png\',dpi=170);plt.close(fig)\nlines=[\'| Features | Model | Accuracy, mean ± SD | Macro F1 |\',\'|---|---|---:|---:|\']\nfor s in r[\'summary\']:lines.append(f"| {s[\'variant\']} | {s[\'model\']} | {s[\'accuracy_mean\']*100:.2f} ± {s[\'accuracy_sd\']*100:.2f}% | {s[\'macro_f1_mean\']:.3f} |")\nPath(\'research/results/TABLES.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\n')


In [ ]:
Path('research/plot_results.py').write_text('from pathlib import Path\nimport json\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nr=json.loads(Path(\'research/results/benchmark.json\').read_text());out=Path(\'docs/assets\');out.mkdir(exist_ok=True)\nplt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'axes.spines.top\':False,\'axes.spines.right\':False,\'figure.facecolor\':\'#f6f7f3\',\'axes.facecolor\':\'#f6f7f3\'})\nmodels=[\'centroid\',\'knn\',\'mlp\'];colors=[\'#a6b5a1\',\'#69a88a\',\'#096b59\']\nfig,ax=plt.subplots(figsize=(9,4.5));base=[x for x in r[\'summary\'] if x[\'variant\']==\'full\'];ax.bar(models,[x[\'accuracy_mean\']*100 for x in base],yerr=[x[\'accuracy_sd\']*100 for x in base],color=colors,capsize=6)\nfor i,x in enumerate(base):ax.text(i,x[\'accuracy_mean\']*100+2,f"{x[\'accuracy_mean\']*100:.1f}%",ha=\'center\',fontweight=\'bold\')\nax.set(ylim=(0,105),ylabel=\'Accuracy (%)\',title=\'UCI hand postures · held-out user-ID evaluation\');fig.text(.1,.01,\'Mean ± SD across 14 IDs. Public Vicon data; not webcam or QOLDA measurements.\',fontsize=9);fig.tight_layout(rect=[0,.06,1,1]);fig.savefig(out/\'benchmark.png\',dpi=170);plt.close(fig)\nfig,ax=plt.subplots(figsize=(9,4.5));variants=[\'full\',\'no_count\',\'no_pair_distances\'];xx=np.arange(3)\nfor i,m in enumerate(models):ax.bar(xx+(i-1)*.23,[next(s for s in r[\'summary\'] if s[\'model\']==m and s[\'variant\']==v)[\'macro_f1_mean\'] for v in variants],width=.23,label=m,color=colors[i])\nax.set_xticks(xx,[\'All features\',\'Without marker count\',\'Without pair distances\']);ax.set(ylabel=\'Macro F1\',ylim=(0,1),title=\'Feature ablation · identical user-ID splits\');ax.legend();fig.tight_layout();fig.savefig(out/\'ablation.png\',dpi=170);plt.close(fig)\nfig,ax=plt.subplots(figsize=(9,4.5))\nfor m,c in zip(models,colors):\n values=[next(s for s in base if s[\'model\']==m)[\'accuracy_mean\']]+[np.mean([s[\'accuracy\'] for s in r[\'synthetic_marker_dropout\'] if s[\'model\']==m and s[\'drop_markers\']==d]) for d in [1,2]]\n ax.plot([0,1,2],np.array(values)*100,\'o-\',label=m,color=c)\nax.set(xticks=[0,1,2],xlabel=\'Artificially removed markers (minimum 3 retained)\',ylabel=\'Accuracy (%)\',title=\'Synthetic occlusion stress test · not measured camera failure\');ax.legend();fig.tight_layout();fig.savefig(out/\'robustness.png\',dpi=170);plt.close(fig)\nlines=[\'| Features | Model | Accuracy, mean ± SD | Macro F1 |\',\'|---|---|---:|---:|\']\nfor s in r[\'summary\']:lines.append(f"| {s[\'variant\']} | {s[\'model\']} | {s[\'accuracy_mean\']*100:.2f} ± {s[\'accuracy_sd\']*100:.2f}% | {s[\'macro_f1_mean\']:.3f} |")\nPath(\'research/results/TABLES.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\n')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"research/benchmark.py"],check=True)
subprocess.run([sys.executable,"research/plot_results.py"],check=True)

In [ ]:
import json
from IPython.display import display, Image, Markdown
r=json.loads(Path("research/results/benchmark.json").read_text())
display(Markdown(Path("research/results/TABLES.md").read_text()))
display(Image("docs/assets/benchmark.png"))
print(r["versions"])

Library versions are printed and saved in the report. The notebook uses the available Colab packages; use research/requirements.txt in Python 3.12 for the recorded environment. Small numerical differences may occur. Synthetic marker-dropout stress tests are labeled separately. No hardware agreement results are claimed.